# FLAN-T5 Fine-Tuning for Legal Q&A — Reference Notebook

> **Reference notebook.** See [`fine_tuning.md`](../06-07-08-transfer-learning-fine-tuning/fine_tuning.md)
> for general fine-tuning concepts,
> [`metrics_to_evaluate_llms.md`](../03-transformers-and-llms-p1/metrics_to_evaluate_llms.md#rouge-recall-oriented-understudy-for-gisting-evaluation)
> for what ROUGE measures, and [`README.md`](./README.md) for the fuller walkthrough this notebook
> distills.

**Methods covered:**
- Loading an encoder-decoder (seq2seq) checkpoint (`AutoModelForSeq2SeqLM` + `AutoTokenizer`)
- Prefixing + tokenizing text-to-text training pairs, with `DataCollatorForSeq2Seq` handling dynamic
  padding and label masking
- Computing **ROUGE** during evaluation via a custom `compute_metrics` function
- Full fine-tuning with `Seq2SeqTrainer`
- Saving and reloading the fine-tuned model for inference (`.generate()`)

**Use this as a reference when:** you need copy-paste-ready code for full (non-PEFT) fine-tuning of a
seq2seq model on question/answer pairs, with ROUGE evaluation wired into the training loop.

**Don't use this as a reference for:** PEFT/LoRA fine-tuning (see the QLoRA notebook in
[module 06-07-08](../06-07-08-transfer-learning-fine-tuning/qlora_sentiment_finetuning.ipynb)) or
decoder-only causal LM fine-tuning.

In [ ]:
import numpy as np
import nltk
import evaluate

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# 80/20 train/test split of question/answer pairs.
dataset = load_dataset("csv", data_files="dataset.csv", delimiter=",")
dataset = dataset["train"].train_test_split(test_size=0.2)

In [ ]:
base_repo = "google/flan-t5-base"

# legacy=False opts into the newer, non-legacy SentencePiece conversion recommended for T5-family models.
tokenizer = AutoTokenizer.from_pretrained(base_repo, legacy=False)
model = AutoModelForSeq2SeqLM.from_pretrained(base_repo)

In [ ]:
# Dynamically pads each batch to its longest sequence and replaces padding token ids in the
# labels with -100, the value cross-entropy loss is configured to ignore.
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [ ]:
# FLAN-T5 was pretrained to condition on instruction-like prefixes -- framing the task in
# natural language before the question itself.
prefix = "answer the question: "

def preprocess(batch):
    inputs = [prefix + question for question in batch["question"]]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)

    # Legal answers run much longer than the questions that prompt them, hence the larger budget.
    labels = tokenizer(text_target=batch["answer"], max_length=512, truncation=True)
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_dataset = dataset.map(preprocess, batched=True)

In [ ]:
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    predictions, labels = eval_preds

    # -100 (the loss-masking value) isn't a valid token id -- restore the real pad id before decoding.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # ROUGE-style preprocessing conventionally operates sentence-by-sentence.
    decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in decoded_labels]

    # use_stemmer reduces words to their stem (e.g. "filing"/"filed" -> "file") so ROUGE doesn't
    # over-penalize legitimate morphological variation in legal phrasing.
    return rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="training_output",
    eval_strategy="epoch",
    learning_rate=3e-4,           # small -- adapting a pretrained model, not training from scratch
    per_device_train_batch_size=4,
    per_device_eval_batch_size=2, # generation during eval is more memory-hungry than a forward pass
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,   # required so eval actually generates text, needed for ROUGE
    push_to_hub=False,
    report_to="none",
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# Saves model weights and tokenizer together -- whoever loads it later gets the exact
# tokenizer it was fine-tuned with.
trainer.save_model("legal_qa_flan_t5")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("legal_qa_flan_t5")
model = AutoModelForSeq2SeqLM.from_pretrained("legal_qa_flan_t5")

question = "Can I move out of state with my children if I have a custody agreement in that state?"
input_ids = tokenizer(question, return_tensors="pt").input_ids

# Sampling with a fairly low temperature -- some lexical variety while staying close to the
# model's most likely continuations.
output_ids = model.generate(input_ids, max_length=50, temperature=0.4, do_sample=True)
answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("Question:", question)
print("Answer:", answer)